# New topo DEMs in Northwest WA
## What data is available?

Under development for the [Cascadia CoPes Hub](https://cascadiacopeshub.org/) project, supported by NSF.

For the tiles shown in the map below, it seems like no version of the .nc files (MHW) exist on the NCEI servers, and for some of the tiles (in the Strait of Juan de Fuca) the .tif files are dated 2021, so there is no 2025 version.  Are those the most recent?

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from clawpack.geoclaw import topotools
from clawpack.visclaw import geoplot
from importlib import reload
import geopandas as gpd
import folium
import pandas as pd

In [ ]:
import find_topo_source

## Select a set of tiles

Specify the NW corner of the 0.25 x 0.25 degree tiles to search for:

In [ ]:
eighth = 1/8  # half tile width, 0.0125 degrees
yrange = array(arange(47.5, 48.5+eighth, 0.25))
xrange = array(arange(-125.00, -124.25+eighth, 0.25))

if 0:
    name = 'NCEI tiles in NW Washington'
    find_topo_source.make_combined_tile_kml(name, xrange, yrange)
    
# make list of tile names (north to south):
tile_names = []
for y in yrange[-1::-1]:
    for x in xrange:
        xm = x+eighth
        ym = y-eighth
        tile_name = find_topo_source.tile_coords(xm,ym,verbose=False)
        #print(f'midpoints: {xm:.3f}, {ym:.3f}, NW corner:  {x:.3f}, {y:.3f}, tile: {tile_name}')
        tile_names.append(tile_name)

## Search for tiles in catlogs:

In [ ]:
print('Versions of tiles found in these catalogs:')
versions = {}
for tile_name in tile_names:
    versions[tile_name] = []
    tile_urls = find_topo_source.find_tile_url(tile_name, verbose=False)
    for url in tile_urls:
        if url[-3:]=='.nc':
            vinfo = url[-9:]
        else:
            vinfo = url[-10:]
        if 'v' not in vinfo:
            vinfo = vinfo[2:]
        versions[tile_name].append(vinfo)
    print(f'    {tile_name}: ', versions[tile_name])

## Map of tiles

In [ ]:
m = folium.Map(location=(47.5,-125), zoom_start=8, height=800, scrollWheelZoom=False)

for y in yrange[-1::-1]:
    for x in xrange:
        xm = x+eighth
        ym = y-eighth
        tile_name = find_topo_source.tile_coords(xm,ym,verbose=False)
        popup = f'<b>{tile_name}</b>'
        for vinfo in versions[tile_name]:
            popup = popup + f'\n{vinfo}'
        folium.Marker(
            location=[y-eighth,x+eighth],
            popup = popup,
            tooltip = "Click for info",
            #tooltip = f"<b>Tile:</b>\n {tile_name}",
            icon=folium.Icon(color="red", icon="info-sign") # Customize the marker's appearance
        ).add_to(m) 
        x1 = x
        x2 = x + 0.25
        y1 = y - 0.25
        y2 = y

        found_nc = array(['.nc' in v for v in versions[tile_name]]).sum() > 0
        found_tif = array(['.tif' in v for v in versions[tile_name]]).sum() > 0
        if found_nc and found_tif:
            color = 'green'
            tip = 'nc and tif'
        elif found_tif:
            color = 'blue'
            tip = 'tif only'
        else:
            color = 'red'
            tip = 'not found'
        
        folium.Polygon(
            locations=[[y1,x1], [y1,x2], [y2,x2], [y2,x1]],
            color="black", # Outline color
            weight=1,
            fill=True,
            fillColor=color,
            fillOpacity=0.2,
            tooltip=tip
        ).add_to(m)

for lat in arange(yrange[0]-0.25, yrange[-1]+.1, 0.25):
    folium.PolyLine(
           [[lat,-125.25], [lat,-123.25]],
           color='black', weight=1, opacity=0.5
       ).add_to(m)

    #Add label at edge
    folium.Marker(
        [lat+0.05, -125.25],
        icon=folium.DivIcon(html=f'<div style="font-size: 12pt; color: gray;">{lat}°</div>')
        ).add_to(m)

# save as stand-alone html file:
fname = 'NoWA-Tiles-Map.html'
m.save(fname)
print('Created ',fname)

# display map inline:
m

The stand-alone map can be viewed at [interactive map](SoWA-Tiles-Map.html).

## Versions of these tiles found on NCEI webpages

Print out more details of which catalogs tiles are found in and urls:

In [ ]:

tile_names = []
for y in yrange[-1::-1]:
    print('\n=====================')
    print(f'Latitude y = {y:.2f}:')
    print('=====================')

    for x in xrange:
        xm = x+eighth
        ym = y-eighth
        tile_name = find_topo_source.tile_coords(xm,ym,verbose=False)
        print('\n------------------------------------------------')
        print(f'Tile: {tile_name},  NW corner:  {x:.3f}, {y:.3f}')
        tile_names.append(tile_name)
        tile_urls = find_topo_source.find_tile_url(tile_name, verbose=True)
        if len(tile_urls) == 0:
            print(f'  {tile_name} not found')
